# EDA — Predicción del Potencial Fotovoltaico Global (`potenciaPV`)

**Proyecto final — Machine Learning**

Dataset base: *Global Solar Power Plants Dataset* — [cimejia/solarPV](https://github.com/cimejia/solarPV)
(Universidad de Alicante · Universidad Central del Ecuador), licencia **CC BY-NC 4.0**.

---

### Objetivo de este notebook

Realizar un Análisis Exploratorio de Datos (EDA) **completo y riguroso** sobre el dataset de 58,978
estaciones/plantas solares, como insumo para construir un modelo de Machine Learning capaz de predecir
el **potencial fotovoltaico anual (`potenciaPV`, kWh/kWp)** en *cualquier ubicación del mundo*, combinando
variables de terreno con datos meteorológicos en tiempo real.

### Variable objetivo (target)

`potenciaPV` — Potencial fotovoltaico anual (kWh/kWp). Corresponde a la columna `pv_potential` /
`POTENCIAL FOTOVOLTÁICO` según la versión del dataset.

### Variables predictoras candidatas

| Variable (ES) | Variable (EN, Google Sheet) | Descripción | Unidad |
|---|---|---|---|
| `latitud` | `latitude` | Latitud | Grados (°) |
| `longitud` | `longitude` | Longitud | Grados (°) |
| `elevacion` | `elevation` | Elevación sobre el nivel del mar | msnm |
| `pendiente` | `slope` | Pendiente del terreno | Grados (°) |
| `aspecto` | `aspect` | Orientación de la pendiente | Grados (°), 0–360 (circular) |
| `curvatura` | `curvature` | Curvatura del terreno | — |
| `distancia` | `dist_to_road` | Distancia a la vía más cercana | m |
| `temperatura` | `ambient_temperature` | Temperatura ambiente | °C |
| `irradiancia` | `ghi` | Irradiancia global horizontal | kWh/m² |
| `humedad` | `humidity` | Humedad relativa | % |
| `v_viento` | `wind_speed` | Velocidad del viento | m/s |
| `d_viento` | `wind_direction` | Dirección del viento | Grados (°), 0–360 (circular) |
| `inclinacion` | `optimal_tilt` | Inclinación óptima del panel | Grados (°) |

### Variables a **excluir** del set de features (riesgo de fuga de datos / redundancia)

- `capacity` — capacidad instalada (MW): no es un insumo físico disponible para un sitio nuevo aún no
  construido; es información posterior a la decisión de ubicación.
- `solar_aptitude`, `solar_aptitude_rounded`, `solar_aptittude_class` — índice compuesto de idoneidad
  calculado por los autores originales; su método de cálculo no está confirmado y podría derivarse en
  parte de `pv_potential`. Se analiza su relación con el target en la Sección 9, pero **no se usan como
  predictoras**.

### Estructura del notebook

1. Configuración e importación de librerías
2. Carga de datos y estandarización de columnas
3. Inspección inicial (shape, tipos, muestra)
4. Calidad de datos (nulos, duplicados, formatos, rangos válidos)
5. Análisis univariado — variable objetivo
6. Análisis univariado — variables predictoras
7. Análisis geográfico y sesgo de cobertura
8. Variables angulares (aspecto, dirección del viento) — tratamiento circular
9. Chequeo de fuga de datos (`solar_aptitude`, `capacity` vs. target)
10. Relación `inclinación óptima` vs. `latitud`
11. Análisis bivariado — features vs. target
12. Correlación y multicolinealidad
13. Detección de outliers
14. Segmentación (estado operacional, tamaño, clima)
15. Conclusiones y recomendaciones para modelado


## 1. Configuración e importación de librerías

In [ ]:
# Manipulación de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px

# Estadística
from scipy import stats
from scipy.stats import skew, kurtosis

# Geoespacial (opcional — usado en la Sección 7)
try:
    import geopandas as gpd
    from shapely.geometry import Point
    GEOPANDAS_OK = True
except ImportError:
    GEOPANDAS_OK = False
    print("geopandas no disponible — la Sección 7 usará solo scatter plots con plotly (no requiere geopandas).")

# Multicolinealidad
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Carga de datos y estandarización de columnas

El dataset puede provenir de dos fuentes equivalentes:

1. **Google Sheets (export CSV)** — la copia de trabajo del grupo, con nombres de columna en inglés
   (`latitude`, `longitude`, `ghi`, `pv_potential`, ...).
2. **CSV local** descargado directamente del repositorio `cimejia/solarPV`, con nombres de columna en
   español (`LATITUD`, `LONGITUD`, `IRRADIANCIA GLOBAL`, `POTENCIAL FOTOVOLTÁICO`, ...).

Para que este notebook funcione sin importar la fuente, se define un **diccionario de normalización**
que traduce cualquiera de los dos esquemas a un set de nombres estándar en español (los mismos usados
en `FEATURE_COLS` del modelo de referencia del repositorio original).


In [ ]:
# --- Fuente de datos ---
# Opción A: Google Sheets (como en el notebook original del grupo)
SHEET_ID = "1hd9qGZoQBRnFKtcrlKAb-CZ4oShwEEHNdgFpJU9V0uo"
SHEET_GID = "1816558602"
SHEET_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={SHEET_GID}"

df = pd.read_csv(SHEET_URL)

In [ ]:
from collections import Counter

In [ ]:
df[]
Counter(df[''])

In [ ]:
df.shape

In [ ]:
# --- Normalización de nombres de columna ---
# Mapea cualquiera de los dos esquemas (EN de Google Sheets / ES del repo original) a un
# esquema estándar único que se usará en todo el resto del notebook.

COLUMN_MAP = {
    # identificación
    "OBJECTID": "objectid", "code": "codigo", "CÓDIGO": "codigo",
    "plant_name": "nombre", "NOMBRE": "nombre",
    "country": "pais", "PAÍS": "pais",
    "operational_status": "estado", "ESTADO": "estado",
    # ubicación
    "longitude": "longitud", "LONGITUD": "longitud",
    "latitude": "latitud", "LATITUD": "latitud",
    "elevation": "elevacion", "ELEVACIÓN": "elevacion",
    "area": "area", "ÁREA": "area",
    "size": "tamano", "TAMAÑO": "tamano",
    # terreno
    "slope": "pendiente", "PENDIENTE": "pendiente",
    "slope_type": "t_pendiente", "T_PENDIENTE": "t_pendiente",
    "curvature": "curvatura", "CURVATURA": "curvatura",
    "curvature_type": "t_curvatura", "T_CURVATURA": "t_curvatura",
    "aspect": "aspecto", "ASPECTO": "aspecto",
    "aspect_type": "t_aspecto", "T_ASPECTO": "t_aspecto",
    "dist_to_road": "distancia", "DISTANCIA A LA VIA": "distancia",
    # clima
    "ambient_temperature": "temperatura", "TEMPERATURA": "temperatura",
    "ghi": "irradiancia", "IRRADIANCIA GLOBAL": "irradiancia",
    "humidity": "humedad", "HUMEDAD": "humedad",
    "wind_speed": "v_viento", "VELOCIDAD DEL VIENTO": "v_viento",
    "wind_direction": "d_viento", "DIRECCIÓN DEL VIENTO": "d_viento",
    "dt_wind": "dt_viento", "DT_VIENTO": "dt_viento",
    # técnico / target / columnas a excluir del modelado
    "solar_aptitude": "aptitud_solar", "APTITUD SOLAR": "aptitud_solar",
    "solar_aptitude_rounded": "aptitud_solar_redondeada",
    "solar_aptittude_class": "t_aptitud_solar", "T_APTITUDS": "t_aptitud_solar",
    "capacity": "capacidad", "CAPACIDAD": "capacidad",
    "optimal_tilt": "inclinacion", "INCLINACIÓN ÓPTIMA": "inclinacion",
    "pv_potential": "potenciaPV", "POTENCIAL FOTOVOLTÁICO": "potenciaPV",
}

df = df_raw.rename(columns=COLUMN_MAP).copy()

# Columnas que esperamos tras la normalización
TARGET_COL = "potenciaPV"
FEATURE_COLS = [
    "latitud", "longitud", "elevacion", "pendiente", "aspecto", "curvatura",
    "distancia", "temperatura", "irradiancia", "humedad", "v_viento",
    "d_viento", "inclinacion",
]
ANGULAR_COLS = ["aspecto", "d_viento"]
EXCLUDED_COLS = ["capacidad", "aptitud_solar", "aptitud_solar_redondeada", "t_aptitud_solar"]
ID_COLS = ["objectid", "codigo", "nombre", "pais", "estado", "tamano"]

missing_expected = [c for c in FEATURE_COLS + [TARGET_COL] if c not in df.columns]
if missing_expected:
    print("ATENCIÓN — columnas esperadas no encontradas tras el mapeo:", missing_expected)
    print("Columnas disponibles:", list(df.columns))
else:
    print("Todas las columnas esperadas fueron encontradas y normalizadas correctamente.")

df.head()


## 3. Inspección inicial

In [ ]:
print(f"Dimensiones del dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas\n")
df.info()


In [ ]:
# Tipos de dato por columna y ejemplo de valor
resumen_tipos = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "ejemplo": df.iloc[0],
    "n_unicos": df.nunique(),
})
resumen_tipos


**Nota esperada:** varias columnas numéricas (`latitud`, `pendiente`, `curvatura`, `humedad`, etc.)
probablemente se cargarán como `object` (texto) en vez de `float`, porque el dataset usa **coma decimal**
(formato regional, ej. `"35,078"` en vez de `35.078`). Esto se corrige en la Sección 4.3.

## 4. Calidad de datos

### 4.1 Valores nulos

In [ ]:
nulos = df.isnull().sum().sort_values(ascending=False)
pct_nulos = (nulos / len(df) * 100).round(2)
tabla_nulos = pd.DataFrame({"n_nulos": nulos, "pct_nulos": pct_nulos})
tabla_nulos = tabla_nulos[tabla_nulos["n_nulos"] > 0]

if tabla_nulos.empty:
    print("No se encontraron valores nulos en el dataset.")
else:
    display(tabla_nulos)

    fig, ax = plt.subplots(figsize=(8, max(1.8, 0.6 * len(tabla_nulos))))

    bars = ax.barh(
        tabla_nulos.index,
        tabla_nulos["pct_nulos"],
        color="#3b6fa0",
        height=0.5,
        zorder=3,
    )

    # Etiqueta de porcentaje al final de cada barra
    for bar, pct, n in zip(bars, tabla_nulos["pct_nulos"], tabla_nulos["n_nulos"]):
        ax.text(
            bar.get_width() + 1.5,
            bar.get_y() + bar.get_height() / 2,
            f"{pct:.2f}%  (n={int(n):,})",
            va="center", ha="left",
            fontsize=10, color="#333333",
        )

    ax.set_xlim(0, 100)
    ax.set_xlabel("% de valores nulos")
    ax.set_ylabel("")
    ax.set_title("Porcentaje de valores nulos por columna", fontsize=12, fontweight="bold", pad=12)

    # Contexto: cuántas columnas del total están completas
    n_total_cols = df.shape[1]
    n_con_nulos = len(tabla_nulos)
    ax.text(
        0.98, 1.12,
        f"{n_con_nulos} de {n_total_cols} columnas presentan valores nulos",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=9, color="#666666", style="italic",
    )

    ax.grid(axis="x", linestyle="--", alpha=0.4, zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(left=False)

    plt.tight_layout()
    plt.savefig("fig_nulos.png", dpi=200, bbox_inches="tight")
    plt.show()

### 4.2 Filas y códigos duplicados

In [ ]:
n_dup_filas = df.duplicated().sum()
n_dup_codigo = df.duplicated(subset=["codigo"]).sum() if "codigo" in df.columns else np.nan

print(f"Filas completamente duplicadas: {n_dup_filas}")
print(f"Códigos de planta duplicados:   {n_dup_codigo}")

if n_dup_codigo and n_dup_codigo > 0:
    display(df[df.duplicated(subset=['codigo'], keep=False)].sort_values('codigo').head(10))


### 4.3 Corrección de formato numérico (coma decimal → punto decimal)

El dataset original usa coma como separador decimal en varias columnas numéricas que Pandas interpreta
como texto. Se convierten explícitamente a `float`, y se registra cuántos valores no pudieron convertirse
(quedarían como `NaN`) para no perder ese diagnóstico silenciosamente.

In [ ]:
NUMERIC_COLS_TO_FIX = [
    c for c in ["latitud", "longitud", "elevacion", "area", "pendiente", "curvatura",
                "aspecto", "distancia", "temperatura", "irradiancia", "humedad",
                "v_viento", "d_viento", "inclinacion", "capacidad", "aptitud_solar",
                "aptitud_solar_redondeada", TARGET_COL]
    if c in df.columns
]

conversion_log = {}
for col in NUMERIC_COLS_TO_FIX:
    if df[col].dtype == object:
        antes_na = df[col].isna().sum()
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(",", ".", regex=False).str.replace(" ", "", regex=False),
            errors="coerce",
        )
        despues_na = df[col].isna().sum()
        conversion_log[col] = despues_na - antes_na
    else:
        df[col] = pd.to_numeric(df[col], errors="coerce")

log_df = pd.DataFrame.from_dict(conversion_log, orient="index", columns=["nuevos_NaN_por_conversion"])
log_df = log_df[log_df["nuevos_NaN_por_conversion"] > 0]

if log_df.empty:
    print("Conversión numérica completada sin generar valores nulos nuevos.")
else:
    print("Columnas donde la conversión introdujo NaN (valores no numéricos / malformados):")
    display(log_df)

df[NUMERIC_COLS_TO_FIX].dtypes


### 4.4 Validación de rangos lógicos / físicos

Se verifica que cada variable respete su rango físicamente válido. Cualquier violación indica errores de
captura o de unidades que deben tratarse antes de modelar.

In [ ]:
rango_esperado = {
    "latitud":     (-90, 90),
    "longitud":    (-180, 180),
    "pendiente":   (0, 90),
    "aspecto":     (0, 360),
    "d_viento":    (0, 360),
    "humedad":     (0, 100),
    "inclinacion": (0, 90),
    "elevacion":   (-430, 8850),   # Mar Muerto -> Everest, margen razonable
    "v_viento":    (0, 60),        # m/s, margen amplio para vientos extremos
    "distancia":   (0, None),
    "irradiancia": (0, None),
    TARGET_COL:    (0, None),
}

violaciones = []
for col, (lo, hi) in rango_esperado.items():
    if col not in df.columns:
        continue
    mask = pd.Series(False, index=df.index)
    if lo is not None:
        mask |= df[col] < lo
    if hi is not None:
        mask |= df[col] > hi
    n_viol = mask.sum()
    if n_viol > 0:
        violaciones.append({"columna": col, "rango_valido": f"[{lo}, {hi}]", "n_violaciones": n_viol,
                             "pct": round(n_viol / len(df) * 100, 3)})

if violaciones:
    display(pd.DataFrame(violaciones))
else:
    print("Todas las columnas verificadas están dentro de su rango físico esperado.")


### 4.5 Consistencia entre variable numérica y su categoría asociada

In [ ]:
# Ej.: 'pendiente' (numérica) vs 't_pendiente' (categórica) deben ser coherentes.
# Se muestra la distribución de la variable numérica por cada categoría, para detectar
# inconsistencias de etiquetado (ej. pendientes altas etiquetadas como 'Plano').
pares_num_cat = [("pendiente", "t_pendiente"), ("curvatura", "t_curvatura"), ("aspecto", "t_aspecto")]

for num_col, cat_col in pares_num_cat:
    if num_col in df.columns and cat_col in df.columns:
        print(f"\n--- {num_col} agrupado por {cat_col} ---")
        display(df.groupby(cat_col)[num_col].describe()[["count", "mean", "min", "max"]])


## 5. Análisis univariado — variable objetivo (`potenciaPV`)

In [ ]:
target = df[TARGET_COL].dropna()

print(f"n = {len(target):,}")
print(target.describe())
print(f"\nSkewness (asimetría): {skew(target):.3f}")
print(f"Kurtosis:              {kurtosis(target):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(target, bins=60, kde=True, ax=axes[0], color="#e07b39")
axes[0].set_title("Distribución de potenciaPV")
axes[0].set_xlabel("kWh/kWp (anual)")

sns.boxplot(x=target, ax=axes[1], color="#e07b39")
axes[1].set_title("Boxplot de potenciaPV")
axes[1].set_xlabel("kWh/kWp (anual)")

plt.tight_layout()
plt.show()


In [ ]:
# Distribución del target por país (top 15 países con más registros)
top_paises = df["pais"].value_counts().head(15).index
df_top = df[df["pais"].isin(top_paises)].copy()

# Conteo de registros por país
n_por_pais = df_top["pais"].value_counts()

# Orden por número de registros (de mayor a menor)
orden_n = n_por_pais.sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 7))

sns.boxplot(
    data=df_top,
    x=TARGET_COL, y="pais",
    order=orden_n,
    hue="pais", palette="viridis", legend=False,
    width=0.6, fliersize=2.5, linewidth=1,
    ax=ax, zorder=3,
)

# Marcador de la media (además de la mediana que ya muestra la caja)
medias = df_top.groupby("pais")[TARGET_COL].mean().reindex(orden_n)
ax.scatter(
    medias.values, range(len(orden_n)),
    marker="D", color="white", edgecolor="black",
    s=35, zorder=4, label="Media",
)

# Etiquetas de tamaño de muestra a la derecha de cada caja
xmax = df_top[TARGET_COL].max()
for i, pais in enumerate(orden_n):
    ax.text(
        xmax * 1.02, i,
        f"n={n_por_pais[pais]:,}",
        va="center", ha="left", fontsize=8.5, color="#555555",
    )

ax.set_title(
    "potenciaPV por país (Top 15 países con más registros)\n"
    "ordenados por número de registros",
    fontsize=12, fontweight="bold", pad=12,
)
ax.set_xlabel("kWh/kWp (anual)")
ax.set_ylabel("")
ax.set_xlim(right=xmax * 1.18)  # espacio para las etiquetas de n
ax.grid(axis="x", linestyle="--", alpha=0.35, zorder=0)
ax.spines[["top", "right"]].set_visible(False)

ax.legend(loc="lower right", frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig("fig_target_pais.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Análisis univariado — variables predictoras

In [ ]:
numeric_features = [c for c in FEATURE_COLS if c in df.columns]

fig, axes = plt.subplots(4, 4, figsize=(16, 13))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.histplot(df[col].dropna(), bins=40, kde=True, ax=axes[i], color="#3b6fa0")
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("")

for j in range(len(numeric_features), len(axes)):
    axes[j].axis("off")

plt.suptitle("Distribución de todas las variables predictoras numéricas", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Tabla resumen estadístico de todas las features numéricas
df[numeric_features].describe().T.assign(
    skewness=df[numeric_features].skew(),
    missing_pct=(df[numeric_features].isna().mean() * 100).round(2),
)


In [ ]:
# Variables categóricas
cat_cols = [c for c in ["pais", "estado", "tamano"] if c in df.columns]

for col in cat_cols:
    n_cat = df[col].nunique()
    print(f"\n{col}: {n_cat} categorías únicas")
    display(df[col].value_counts().head(10).to_frame("count"))

fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 4.5))
if len(cat_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, cat_cols):
    top = df[col].value_counts().head(10)
    sns.barplot(x=top.values, y=top.index, ax=ax, color="#5b8c5a")
    ax.set_title(f"Top categorías — {col}")
plt.tight_layout()
plt.show()


## 7. Análisis geográfico y sesgo de cobertura

Dado que el objetivo final del proyecto es generalizar a **cualquier ubicación del mundo**, es crítico
cuantificar qué tan sesgada está la cobertura geográfica del dataset — un modelo entrenado con datos
concentrados en pocos países puede generalizar mal fuera de esas zonas.

In [ ]:
conteo_paises = df["pais"].value_counts()
top10 = conteo_paises.head(10)
pct_top10 = top10.sum() / len(df) * 100

print(f"Los 10 países con más registros concentran el {pct_top10:.1f}% del dataset:")
display(top10.to_frame("n_plantas"))
print(f"\nTotal de países representados: {df['pais'].nunique()}")


In [ ]:
# Mapa mundial de dispersión de plantas, coloreado por potencial fotovoltaico
fig = px.scatter_geo(
    df.dropna(subset=["latitud", "longitud", TARGET_COL]),
    lat="latitud", lon="longitud",
    color=TARGET_COL,
    color_continuous_scale="Viridis",
    hover_name="pais",
    opacity=0.5,
    title="Distribución global de plantas solares — color = potenciaPV (kWh/kWp)",
)
fig.update_layout(height=550, margin=dict(l=0, r=0, t=40, b=0))
fig.show()


In [ ]:
# Agregación por país: cobertura vs. potencial promedio
agg_pais = df.groupby("pais").agg(
    n_plantas=("codigo", "count"),
    potenciaPV_media=(TARGET_COL, "mean"),
    potenciaPV_std=(TARGET_COL, "std"),
).sort_values("n_plantas", ascending=False)

agg_pais.head(20)


**Punto de atención para el diseño del modelo:** dado el sesgo hacia China, Japón, EE.UU., Alemania
e India, se recomienda usar **validación cruzada espacial (por país/continente)**, en lugar de un split
aleatorio, para obtener una estimación realista de qué tan bien generalizará el modelo a ubicaciones
nuevas — un split aleatorio tenderá a sobreestimar el desempeño porque el modelo "memoriza" patrones
regionales sobrerrepresentados.

## 8. Variables angulares (`aspecto`, `d_viento`)

Estas variables son **circulares** (0° y 359° representan direcciones casi idénticas), por lo que un
histograma lineal estándar puede ser engañoso, y no deben alimentarse directamente a un modelo como
variables numéricas lineales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={"projection": "polar"})

for ax, col in zip(axes, ANGULAR_COLS):
    data = np.deg2rad(df[col].dropna())
    ax.hist(data, bins=36, color="#8a4fbf", alpha=0.75)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(col)

plt.suptitle("Distribución circular de variables angulares", y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
# Demostración del encoding cíclico (seno/coseno) que se usará en la etapa de modelado
for col in ANGULAR_COLS:
    rad = np.deg2rad(df[col])
    df[f"{col}_sin"] = np.sin(rad)
    df[f"{col}_cos"] = np.cos(rad)

df[ANGULAR_COLS + [f"{ANGULAR_COLS[0]}_sin", f"{ANGULAR_COLS[0]}_cos"]].head()


## 9. Chequeo de fuga de datos (`capacidad`, `aptitud_solar` vs. `potenciaPV`)

`aptitud_solar` (solar_aptitude) y `capacidad` (capacity) **no forman parte del set de features** definido
en la Sección 2, pero es importante documentar su relación con el target — tanto para justificar la
exclusión como porque el equipo podría querer usarlas como variables de referencia/contexto en el paper
(nunca como predictoras).

In [ ]:
cols_leakage_check = [c for c in ["capacidad", "aptitud_solar"] if c in df.columns]

if cols_leakage_check:
    corr_leakage = df[cols_leakage_check + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
    print("Correlación de Pearson con potenciaPV:")
    display(corr_leakage.to_frame("correlacion"))

    fig, axes = plt.subplots(1, len(cols_leakage_check), figsize=(6 * len(cols_leakage_check), 4.5))
    if len(cols_leakage_check) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols_leakage_check):
        sns.scatterplot(data=df.sample(min(5000, len(df)), random_state=RANDOM_STATE),
                         x=col, y=TARGET_COL, alpha=0.3, ax=ax, color="#c0392b")
        ax.set_title(f"{col} vs. {TARGET_COL}")
    plt.tight_layout()
    plt.show()

    print(
        "\nRegla de decisión: si |correlación| > 0.85 con el target, la variable se considera "
        "de alto riesgo de fuga y se mantiene excluida del set de modelado (ya excluida por diseño "
        "en este notebook)."
    )
else:
    print("Columnas de aptitud/capacidad no encontradas en este dataset — nada que verificar.")


## 10. Relación entre `inclinacion` (tilt óptimo) y `latitud`

Se espera una relación cuasi-determinística (regla empírica común: inclinación óptima ≈ |latitud|).
Confirmar esto es valioso porque, de cumplirse, el equipo puede **generar esta feature analíticamente**
para cualquier ubicación nueva, sin depender de que esté presente en el dataset original — un requisito
clave para el pipeline de inferencia en tiempo real.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sample = df.dropna(subset=["latitud", "inclinacion"]).sample(min(8000, len(df)), random_state=RANDOM_STATE)

sns.scatterplot(data=sample, x="latitud", y="inclinacion", alpha=0.25, ax=ax, color="#2f6f4f")
# línea de referencia inclinacion = |latitud|
lat_range = np.linspace(sample["latitud"].min(), sample["latitud"].max(), 100)
ax.plot(lat_range, np.abs(lat_range), color="black", linestyle="--", label="Referencia: inclinación = |latitud|")

ax.set_title("Inclinación óptima vs. Latitud")
ax.legend()
plt.tight_layout()
plt.show()

corr_tilt_lat = df[["latitud", "inclinacion"]].corr().iloc[0, 1]
print(f"Correlación (latitud, inclinación): {corr_tilt_lat:.3f}")
print(f"Correlación (|latitud|, inclinación): {df['latitud'].abs().corr(df['inclinacion']):.3f}")


## 11. Análisis bivariado — variables predictoras vs. `potenciaPV`

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
axes = axes.flatten()
sample = df.sample(min(8000, len(df)), random_state=RANDOM_STATE)

plot_features = [c for c in numeric_features if c not in ["latitud", "longitud"]]
for i, col in enumerate(plot_features):
    sns.scatterplot(data=sample, x=col, y=TARGET_COL, alpha=0.25, ax=axes[i], s=15, color="#3b6fa0")
    axes[i].set_title(f"{col} vs. potenciaPV", fontsize=10)

for j in range(len(plot_features), len(axes)):
    axes[j].axis("off")

plt.suptitle("Relación de cada feature con la variable objetivo", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()


## 12. Correlación y multicolinealidad

In [ ]:
cols_corr = numeric_features + [TARGET_COL]
corr_matrix = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Matriz de correlación — features numéricas + potenciaPV")
plt.tight_layout()
plt.show()


In [ ]:
print("Correlación de cada feature con el target, ordenada por magnitud:\n")
display(
    corr_matrix[TARGET_COL].drop(TARGET_COL)
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .to_frame("correlacion_con_potenciaPV")
)


### Factor de Inflación de Varianza (VIF)

Mide multicolinealidad entre predictoras (no con el target). Un VIF > 10 suele indicar redundancia
significativa que puede requerir eliminar o combinar variables antes de modelar.

In [ ]:
vif_data = df[numeric_features].dropna()

vif_df = pd.DataFrame({
    "feature": numeric_features,
    "VIF": [variance_inflation_factor(vif_data.values, i) for i in range(len(numeric_features))],
}).sort_values("VIF", ascending=False)

vif_df


## 13. Detección de outliers (método IQR)

In [ ]:
def resumen_outliers_iqr(serie, factor=1.5):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - factor * iqr, q3 + factor * iqr
    outliers = serie[(serie < lo) | (serie > hi)]
    return {
        "n_outliers": len(outliers),
        "pct_outliers": round(len(outliers) / serie.notna().sum() * 100, 2),
        "limite_inferior": round(lo, 3),
        "limite_superior": round(hi, 3),
    }

outlier_summary = pd.DataFrame({
    col: resumen_outliers_iqr(df[col].dropna()) for col in numeric_features + [TARGET_COL]
}).T.sort_values("pct_outliers", ascending=False)

outlier_summary


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 13))
axes = axes.flatten()

for i, col in enumerate(numeric_features + [TARGET_COL]):
    sns.boxplot(x=df[col], ax=axes[i], color="#e07b39")
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("")

for j in range(len(numeric_features) + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Boxplots de todas las variables numéricas (detección visual de outliers)", y=1.01)
plt.tight_layout()
plt.show()


## 14. Segmentación

In [ ]:
# potenciaPV por estado operacional
if "estado" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    orden_estado = df.groupby("estado")[TARGET_COL].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=TARGET_COL, y="estado", order=orden_estado, ax=ax, palette="viridis")
    ax.set_title("potenciaPV por estado operacional de la planta")
    plt.tight_layout()
    plt.show()


In [ ]:
# potenciaPV por tamaño de planta
if "tamano" in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=df, x="tamano", y=TARGET_COL, ax=ax, palette="viridis")
    ax.set_title("potenciaPV por tamaño de planta")
    plt.tight_layout()
    plt.show()


In [ ]:
# Zonas climáticas aproximadas por banda de latitud
bins_lat = [-90, -66.5, -23.5, 0, 23.5, 66.5, 90]
labels_lat = ["Polar S", "Templada S", "Trop. S", "Trop. N", "Templada N", "Polar N"]
df["banda_climatica"] = pd.cut(df["latitud"], bins=bins_lat, labels=labels_lat)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="banda_climatica", y=TARGET_COL, ax=ax, palette="viridis")
ax.set_title("potenciaPV por banda climática (latitud)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 15. Conclusiones y recomendaciones para el modelado

*(Completar cada punto con los resultados numéricos concretos obtenidos al ejecutar el notebook con los
datos reales — esta sección deja la estructura lista para la discusión en el paper.)*

### 15.1 Calidad de datos
- [ ] Resumen de nulos encontrados y estrategia de imputación/eliminación.
- [ ] Duplicados encontrados y tratamiento aplicado.
- [ ] Confirmación de que la conversión de coma a punto decimal no introdujo pérdida significativa de datos.
- [ ] Violaciones de rango físico encontradas (si las hay) y su resolución.

### 15.2 Variable objetivo
- [ ] Forma de la distribución de `potenciaPV` (¿normal, sesgada?) — define si se necesita una
  transformación (ej. log) antes de modelar.
- [ ] Diferencias relevantes de `potenciaPV` por país/región.

### 15.3 Sesgo geográfico
- [ ] Porcentaje de concentración en los países dominantes.
- [ ] Decisión de estrategia de validación (recomendado: **spatial cross-validation** por país/continente,
  en lugar de split aleatorio, dado el objetivo de generalización global).

### 15.4 Variables angulares
- [ ] Confirmación de que `aspecto` y `d_viento` requieren encoding cíclico (seno/coseno) para el modelo.

### 15.5 Fuga de datos
- [ ] Correlación de `capacidad` y `aptitud_solar` con el target — confirmación de exclusión del set de
  features.

### 15.6 Relación `inclinacion` vs. `latitud`
- [ ] Confirmación de si la relación es suficientemente fuerte para calcular `inclinacion` analíticamente
  en ubicaciones nuevas (sin depender del dataset original) — esto es clave para el pipeline de inferencia
  en tiempo real con datos meteorológicos externos.

### 15.7 Multicolinealidad
- [ ] Variables con VIF > 10 — decisión sobre exclusión, combinación (ej. PCA) o regularización
  (Ridge/Lasso) en el modelo.

### 15.8 Outliers
- [ ] Variables con mayor proporción de outliers y decisión de tratamiento (winsorización, transformación,
  o modelos robustos a outliers como Random Forest/Gradient Boosting).

### 15.9 Próximos pasos (fuera del alcance de este EDA)
1. Selección de la API meteorológica en tiempo real (ej. NASA POWER, Open-Meteo) y verificación de que sus
   variables sean compatibles en nombre, unidad y granularidad con las de este dataset.
2. Diseño del split de validación espacial.
3. Ingeniería de features: encoding cíclico de variables angulares, cálculo analítico de `inclinacion`.
4. Selección y comparación de modelos (baseline lineal, Random Forest/XGBoost, y red neuronal como
   referencia directa al trabajo original de `cimejia/solarPV`).
5. Definición de métricas de evaluación (RMSE, MAE, R²) y su interpretación en el contexto físico
   (kWh/kWp).
